<h4> Creating a neural network using torch.nn module </h4>

In [4]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os
import torch.nn as nn

class Model(nn.Module):

    def __init__(self, num_features):
        super().__init__() #invoking constructor of the parent class

        self.linear=nn.Linear(num_features,1) #Format(number_of_input, numberof_output)
        self.sigmoid=nn.Sigmoid()

    def forward(self,features):
        out=self.linear(features) #calculate wx+b
        out=self.sigmoid(out)
        return out

print("Files Python can see:", os.listdir('.'))
from google.colab import drive # type: ignore
drive.mount(r'/content/drive/')

# Important Parameters
learning_rate = 0.1
epochs = 25

# Load Data
df = pd.read_csv(r"/content/drive/MyDrive/DL_CSV/breast-cancer.csv")

# FIXED: Saved the dataframe modification back to df so 'id' is removed
df = df.drop(['id'], axis='columns', errors='ignore')

Y = df['diagnosis']
X = df.drop(['diagnosis'], axis='columns')

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)

# Scaling the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Label Encoding
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

# Converting Numpy Arrays to tensors
#Keep in mind the data types, self.linear = nn.Linear(num_features, 1) creates weights with dtype torch.float32 by default and the dataset was of datatype double
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).unsqueeze(1).float()
y_test_tensor = torch.from_numpy(y_test).unsqueeze(1).float()

model=Model(X_train_tensor.shape[1])

#Adding Loss Function
loss_function=nn.BCELoss()

optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)

for i in range(epochs):
    optimizer.zero_grad()

    #Running Forward Pass
    y_pred=model(X_train_tensor)

    #Calculating Loss
    loss=loss_function(y_pred,y_train_tensor)

    #Running backward propagation
    loss.backward()

    #Updating weights
    optimizer.step()

#Calculating Accuracy
with torch.no_grad():
    pred=model(X_test_tensor)
    pred = (pred > 0.5).float()  # matches float64/double tensor type
    accuracy = (pred == y_test_tensor).float().mean()
    print(f"Accuracy: {accuracy.item() * 100:.2f}%")

Files Python can see: ['.config', 'drive', 'sample_data']
Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
Accuracy: 95.61%
